# Multiobjective binding optimisation for tumour selectivity (v5)
**One full ODE model; a Pareto set of four-rate designs, not a binding-event surrogate.**

The analytical notebook optimised a different, reduced objective and is not used to restrict this search. Here every candidate is judged by intracellular payload exposure and irreversible cell kill in the full V4 model. The design variables remain the two binding on-rates and two off-rates. Affinities are calculated afterwards.

**Plan:** retain the full biological model and declared inputs → optimise tumour and healthy-cell outcomes together → refine several trade-offs → reintegrate feasible candidates → retain the nondominated joint combinations → allow selection using explicit requirements.

Missing biological inputs were assigned synthetic values by an OpenAI language model on 17 November 2026, not measured experimentally. The model demonstrates a reusable method; it does not provide experimentally established affinity recommendations. V1's calibrated physiology and symmetric surface-dissociation assumption are retained without refitting. All code is self-contained; results go to `results_v5/`.

In [ ]:
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import itertools, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from platypus import Problem, Real, Solution, InjectedPopulation, NSGAII
from IPython.display import display, Markdown

# Hours; solution and intracellular concentrations in uM; amounts in umol;
# surface/endosomal states in units of 1e-10 mol/dm2.
F = np.array([.0003, .79, .10, .10, .01])
NAMES = ['T', 'H0', 'H1', 'H2', 'H12']
NCELL = 7043996587630.85
VI = 2.689720494498626e-13
AREA = 3.141592653589793e-8
CAP = NCELL * F * AREA * 1e-4  # umol per scaled surface unit
CELLVOL = NCELL * F * VI     # L
GAIN = AREA * 1e-4 / VI      # uM per scaled surface unit
VC, VP, QA = 11.3, 41.5, 1.05
CLA = 0.015540799881601949
BOUNDS = [(2.,7.),(-5.,2.),(2.,7.),(-5.,2.)]
LIMIT = .0095
RATES = ['kon1','koff1','kon2','koff2']

@dataclass(frozen=True)
class Scenario:
    rho: tuple = (1., 1.)  # multiples of 1e-10 mol/dm2
    k0: tuple = (2., 2.)
    ki: tuple = (2., 2., 2.)
    half: float = np.inf
    permeability: float = 0.  # PS / VI, h^-1; bidirectional, K=fu=1
    payload_volume: float = 52.8  # one WELL-MIXED extracellular pool, exploratory
    payload_clearance: float = 5.28  # L/h; clearance rate 0.1 h^-1
    metabolism: float = 0.
    tau: float = 0.  # 0: instantaneous internal degradation and release
    recycle: float = 0.
    process_share: float = .5  # remaining exit split between processing/degradation
    closure: str = 'symmetric'
    surface_factor: float = 1.
    dose: float = 4.  # umol
    target_densities: tuple = ()  # optional 5 x 2 matrix, multiples of 1e-10 mol/dm2
    fractions: tuple = (.0003,.79,.10,.10,.01)
    total_cells: float = NCELL
    intracellular_volume: float = VI
    cell_area: float = AREA
    antibody_vc: float = VC
    antibody_vp: float = VP
    antibody_q: float = QA
    antibody_clearance: float = CLA
    infusion_h: float = 2.
    horizon_h: float = 24.
    recycling_rates: tuple = ()
    processing_rates: tuple = ()
    degradation_rates: tuple = ()


def density(s):
    if s.target_densities:
        return np.asarray(s.target_densities, dtype=float)
    return np.array([[1,1],[0,0],[.25,0],[0,.25],[.25,.25]]) * s.rho


def make_rhs(theta, s, on):
    CAP=s.total_cells*np.array(s.fractions)*s.cell_area*1e-4
    CELLVOL=s.total_cells*np.array(s.fractions)*s.intracellular_volume
    GAIN=s.cell_area*1e-4/s.intracellular_volume
    VC,VP,QA,CLA=s.antibody_vc,s.antibody_vp,s.antibody_q,s.antibody_clearance
    a1,b1,a2,b2 = 10.**np.asarray(theta) * 3600
    l = 2.4e13 * 1e-10 * 3600 * s.surface_factor
    corr = np.sqrt(a2/a1) if s.closure == 'symmetric' else 1.
    q1,q2 = b1*corr,b2/corr
    kd = np.log(2)/s.half
    rho,k0,ki = density(s), np.array(s.k0), np.array(s.ki)
    kr = s.recycle/s.tau if s.tau else 0.
    kp = (1-s.recycle)*s.process_share/s.tau if s.tau else 0.
    kg = (1-s.recycle)*(1-s.process_share)/s.tau if s.tau else 0.
    if s.recycling_rates:
        kr,kp,kg=map(np.asarray,(s.recycling_rates,s.processing_rates,s.degradation_rates))
    def fun(t,y):
        # 0:4 Ic,Ip,Nc,Np; 4 extracellular payload; 5 Ab lost; 6 payload lost.
        # per cell: R(2), intact S(3), naked S(3), intact E(3), naked E(3),
        # P instantaneous, A cumulative productive targeted release (both uM).
        z=y[7:].reshape(5,16); dz=np.zeros_like(z); out=np.zeros_like(y)
        R=z[:,:2]; S=z[:,2:8].reshape(5,2,3); E=z[:,8:14].reshape(5,2,3)
        dS=dz[:,2:8].reshape(5,2,3); dE=dz[:,8:14].reshape(5,2,3)
        dz[:,:2]=k0*(rho-R)
        for x,c in enumerate([y[0],y[2]]):
            b=S[:,x,:]
            v1=a1*c*1e-6*R[:,0]-b1*b[:,0]
            v2=a2*c*1e-6*R[:,1]-b2*b[:,1]
            w1=l*b[:,1]*R[:,0]-q1*b[:,2]
            w2=l*b[:,0]*R[:,1]-q2*b[:,2]
            dz[:,0]-=v1+w1; dz[:,1]-=v2+w2
            dS[:,x,:]=np.column_stack([v1-w2,v2-w1,w1+w2])-ki*b
            cix=2*x; pix=cix+1
            flow=QA*(y[cix]-y[pix])
            out[cix]=(-CLA*y[cix]-flow-CAP@(v1+v2))/VC
            out[pix]=flow/VP
        shed=kd*S[:,0,:]
        dS[:,0,:]-=shed; dS[:,1,:]+=shed
        out[0]-=kd*y[0]; out[1]-=kd*y[1]
        out[2]+=kd*y[0]; out[3]+=kd*y[1]
        out[0]+=(s.dose/s.infusion_h/VC if on else 0.)
        if s.tau or s.recycling_rates:
            dS[:]+=kr*E
            dE[:,0,:]=ki*S[:,0,:]-(kr+kp+kg)*E[:,0,:]
            dE[:,1,:]=ki*S[:,1,:]+kp*E[:,0,:]-(kr+kg)*E[:,1,:]
            release=((kp+kg)*E[:,0,:]).sum(axis=1)
            ab_loss=(kg*E.sum(axis=1)).sum(axis=1)
        else:
            release=(ki*S[:,0,:]).sum(axis=1)
            ab_loss=(ki*S.sum(axis=1)).sum(axis=1)
        production=GAIN*release
        exchange=s.permeability*(y[4]-z[:,14])
        dz[:,14]=production+exchange-s.metabolism*z[:,14]
        dz[:,15]=production
        # Both antibody compartments release to the explicitly pooled extracellular
        # payload volume; this is a lumping assumption, not central-only transport.
        generated=kd*(VC*y[0]+VP*y[1])+CAP@shed.sum(axis=1)
        out[4]=(generated-s.payload_clearance*y[4]-CELLVOL@exchange)/s.payload_volume
        out[5]=CLA*(y[0]+y[2])+CAP@ab_loss
        out[6]=CLA*y[0]+s.payload_clearance*y[4]+s.metabolism*(CELLVOL@z[:,14])
        out[7:]=dz.ravel()
        return out
    return fun




## 1. Inputs and their origins
The five populations follow Table 1: tumour, healthy with neither target, target 1 only, target 2 only, and both targets. Healthy weights sum to one; the tumour multiplier is additional, as in V1. Densities can be entered independently for each population.

All rates in the input table are per hour except the four binding rates (per second and M⁻¹s⁻¹). Three-element arrays refer to target-1-bound, target-2-bound and both-target-bound complexes. Free-receptor turnover means effective receptor loss; synthesis maintains the specified drug-free density.

We preserve DAR = 1. Released payload is assumed active with unit partition ratio, and both antibody species share binding kinetics. Extracellular payload is one well-mixed pool; this is not a spatial tissue model.

In [ ]:
OUT=Path.cwd()/'results_v5'; OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
@dataclass(frozen=True)
class KillModel:
    c50_uM: tuple = (4.,4.,4.,4.,4.)
    maximum_rate_h: tuple = (.08,.08,.08,.08,.08)

# Explicit rates replace tau/branching shortcuts in the combined model.
BIOLOGY=Scenario(ki=(.2,4.,2.), half=96., permeability=.1, metabolism=.01,
                 recycling_rates=(.6,.6,.6), processing_rates=(.7,.7,.7),
                 degradation_rates=(.7,.7,.7))
PD=KillModel()
INPUT_SET=[('Nominal',BIOLOGY,PD),
 ('Variant A',replace(BIOLOGY,ki=(.16,3.2,1.6),half=80.,permeability=.12),
  replace(PD,c50_uM=(4.8,3.2,3.2,3.2,3.2))),
 ('Variant B',replace(BIOLOGY,ki=(.24,4.8,2.4),half=120.,permeability=.08),
  replace(PD,c50_uM=(3.2,4.8,4.8,4.8,4.8)))]
GLOBAL_LIMIT=.0095
GROUP_LIMITS=np.repeat(.10,4)
SEEDS=(8407,2026)
GLOBAL_EVALUATIONS=256
LOCAL_ITERATIONS=40

paper_fields={'rho','k0','dose','fractions','infusion_h'}
reconstruction_fields={'total_cells','intracellular_volume','cell_area','antibody_vc','antibody_vp',
 'antibody_q','antibody_clearance','closure','surface_factor','horizon_h'}
provenance=[]
for key,val in asdict(BIOLOGY).items():
 origin=('Paper / Table 1; adopted in V1' if key in paper_fields else
         'V1 reconstruction / reference-informed assumption' if key in reconstruction_fields else
         'LLM-synthetic input or structural assumption')
 provenance.append(dict(parameter=key,value=str(val),origin=origin))
provenance += [dict(parameter='KillModel.'+k,value=str(v),origin='LLM-synthetic; requires exposure-response calibration') for k,v in asdict(PD).items()]
provenance += [dict(parameter='Subgroup kill ceilings',value=str(GROUP_LIMITS.tolist()),origin='Illustrative design requirement; not a safety standard'),
 dict(parameter='Binding-rate bounds',value=str(BOUNDS),origin='V1 exploratory bounds, not demonstrated engineering limits')]
units={'rho':'1e-10 mol/dm²','k0':'h⁻¹','ki':'h⁻¹','half':'h','permeability':'h⁻¹',
 'payload_volume':'L','payload_clearance':'L/h','metabolism':'h⁻¹','tau':'h','recycle':'fraction',
 'process_share':'fraction','surface_factor':'multiplier','dose':'µmol','target_densities':'1e-10 mol/dm²',
 'fractions':'population multipliers','total_cells':'cells','intracellular_volume':'L/cell','cell_area':'dm²/cell',
 'antibody_vc':'L','antibody_vp':'L','antibody_q':'L/h','antibody_clearance':'L/h','infusion_h':'h','horizon_h':'h',
 'recycling_rates':'h⁻¹','processing_rates':'h⁻¹','degradation_rates':'h⁻¹','KillModel.c50_uM':'µM',
 'KillModel.maximum_rate_h':'h⁻¹','Subgroup kill ceilings':'fraction','Near-optimal tolerance':'percentage point',
 'Binding-rate bounds':'log10 of M⁻¹s⁻¹ / s⁻¹'}
for row in provenance:
    row['unit']=units.get(row['parameter'],'—')
    if row['parameter'] in ('tau','recycle','process_share'):
        row['origin']='Inactive compatibility setting; explicit three-rate arrays are used'
    if row['parameter']=='target_densities':
        row['origin']='Optional override; empty uses paper/Table 1 density pattern'
provenance=pd.DataFrame(provenance)
provenance.to_csv(OUT/'input_provenance.csv',index=False)
configuration={'biological_inputs':[{ 'name':name,'biology':asdict(s),'kill_model':asdict(pd_)} for name,s,pd_ in INPUT_SET],
 'overall_healthy_limit':GLOBAL_LIMIT,'healthy_group_limits':GROUP_LIMITS.tolist(),
 'log10_binding_bounds':BOUNDS,
 'seeds':SEEDS,'global_evaluations':GLOBAL_EVALUATIONS,'local_iterations':LOCAL_ITERATIONS,
 'synthetic_provenance':'Missing inputs assigned by an OpenAI language model, 2026-11-17. Not measured data.'}
(OUT/'inputs.json').write_text(json.dumps(configuration,indent=2))
display(provenance)
display(pd.DataFrame([{'Input variant':n,'Internalisation /h':s.ki,'Linker half-life h':s.half,
 'Membrane exchange /h':s.permeability,'C50 µM (T,H0,H1,H2,H12)':p.c50_uM} for n,s,p in INPUT_SET]))

configuration['objectives']=['Maximise minimum tumour cell kill','Minimise maximum overall healthy cell kill','Minimise maximum healthy subgroup cell kill']
configuration['epsilon_budgets_percent']=[(.05,1.),(.10,2.5),(.20,5.),(.40,7.5),(.95,10.),(.10,10.)]
(OUT/'inputs.json').write_text(json.dumps(configuration,indent=2))


## 2. From payload concentration to irreversible cell kill
For population j, integrate **dHⱼ/dt = kmaxⱼ Pⱼ/(C50ⱼ + Pⱼ)** and calculate **cell kill = 100[1 − exp(−Hⱼ)]**. This accumulates damage over exposure, so cell kill cannot decrease when payload leaves a cell. C50 and kmax are synthetic; using 4 µM here does not transfer the paper's endpoint potency calibration.

PK, receptor capacity and population weights remain fixed as cell kill accumulates: no cell-depletion feedback, regrowth, or tumour evolution. This is an exposure-to-kill scoring model, not a longitudinal tumour-burden model. Intracellular AUC is integrated as a separate diagnostic.

Surface binding and intact/naked conversion follow the conserved V3 extension. Recycling returns complexes and receptors; processing releases payload while leaving antibody able to recycle; degradation removes antibody and attached receptors. The symmetric surface dissociation closure is inherited from V1, not uniquely established by biology.

In [ ]:
def check_inputs(s,p):
    assert s.closure == "symmetric", "This method retains V1 symmetric surface closure"
    assert np.asarray(density(s)).shape==(5,2) and np.all(density(s)>=0)
    assert len(s.fractions)==5 and np.all(np.array(s.fractions)>=0)
    assert np.isclose(sum(s.fractions[1:]),1.)
    assert 0<s.infusion_h<s.horizon_h and s.dose>=0
    assert min(s.total_cells,s.intracellular_volume,s.cell_area,s.antibody_vc,s.antibody_vp,s.payload_volume)>0
    assert min(s.antibody_q,s.antibody_clearance,s.payload_clearance,s.permeability,s.metabolism)>=0
    assert s.half>0 and np.all(np.array(s.ki)>=0) and np.all(np.array(s.k0)>=0)
    for a in (s.recycling_rates,s.processing_rates,s.degradation_rates):
        assert not a or (len(a)==3 and np.all(np.array(a)>=0))
    assert all(bool(a)==bool(s.recycling_rates) for a in (s.processing_rates,s.degradation_rates))
    assert len(p.c50_uM)==len(p.maximum_rate_h)==5
    assert np.all(np.array(p.c50_uM)>0) and np.all(np.array(p.maximum_rate_h)>=0)

def integrate(theta,s=BIOLOGY,p=PD,tight=False,points=3,method='BDF'):
    check_inputs(s,p)
    y0=np.zeros(97)  # 87 material states + 5 cumulative hazards + 5 AUCs
    y0[7:87].reshape(5,16)[:,:2]=density(s)
    tt=[]; yy=[]
    times=np.unique(np.r_[np.linspace(0,s.horizon_h,points),s.infusion_h])
    for start,end,on in [(0,s.infusion_h,True),(s.infusion_h,s.horizon_h,False)]:
        material=make_rhs(theta,s,on)
        def rhs(t,y):
            d=np.zeros_like(y);d[:87]=material(t,y[:87])
            conc=y[7:87].reshape(5,16)[:,14]
            d[87:92]=np.array(p.maximum_rate_h)*conc/(np.array(p.c50_uM)+conc)
            d[92:97]=conc
            return d
        sol=solve_ivp(rhs,(start,end),y0,method=method,rtol=1e-8 if tight else 2e-6,
                      atol=1e-11 if tight else 2e-9,dense_output=True)
        if not sol.success:raise RuntimeError(sol.message)
        ts=times[(times>=start)&(times<=end)] if start==0 else times[(times>start)&(times<=end)]
        tt.extend(ts);yy.extend(sol.sol(ts).T);y0=sol.y[:,-1]
    y=np.array(yy);t=np.array(tt);z=y[:,7:87].reshape(-1,5,16)
    if y.min() < -2e-6:
        if not tight:return integrate(theta,s,p,True,points,method)
        if method=='BDF':return integrate(theta,s,p,True,points,'Radau')
        raise RuntimeError('Unresolved negative state')
    kill=-np.expm1(-y[:,87:92])
    cap=s.total_cells*np.array(s.fractions)*s.cell_area*1e-4
    vols=s.total_cells*np.array(s.fractions)*s.intracellular_volume
    ab=s.antibody_vc*(y[:,0]+y[:,2])+s.antibody_vp*(y[:,1]+y[:,3])+z[:,:,2:14].sum(axis=2)@cap+y[:,5]
    payload=s.antibody_vc*y[:,0]+s.antibody_vp*y[:,1]+(z[:,:,2:5]+z[:,:,8:11]).sum(axis=2)@cap+s.payload_volume*y[:,4]+z[:,:,14]@vols+y[:,6]
    dose=s.dose*np.minimum(t/s.infusion_h,1.)
    mass=max(abs(ab-dose).max(),abs(payload-dose).max())/max(s.dose,1.)
    if mass>2e-6:raise RuntimeError(f'Mass error exceeds tolerance: {mass}')
    return dict(t=t,y=y,z=z,kill=kill,healthy=kill[:,1:]@np.array(s.fractions[1:]),mass=mass)

def assess(theta,tight=False):
    rows=[]
    for name,s,p in INPUT_SET:
        r=integrate(theta,s,p,tight=tight); k=r['kill'][-1]
        rows.append([k[0],r['healthy'][-1],*k[1:]])
    a=np.array(rows)
    # Every constraint applies to every listed input variant, not an average variant.
    margins=np.r_[GLOBAL_LIMIT-a[:,1],(GROUP_LIMITS-a[:,2:]).ravel()]
    return dict(tumour=100*a[:,0].min(),healthy=100*a[:,1].max(),
                margins=100*margins,outcomes=100*a)


## 3. Check the implementation before searching
These checks concern computation, not experimental accuracy. The immediate-release limit recovers V1's calibrated uptake anchors and Emax output. All actual design searches use the combined model.

In [ ]:
test_theta=np.log10([1e4,1.,1e4,1.])
check_rows=[]
# V1 calibrated anchors: immediate degradation/release, no payload loss.
for label,theta,anchor in [('Figure 4',np.log10([4.6e5,3.4e-4]*2),56.),('Figure 6',test_theta,280.)]:
    r=integrate(theta,Scenario(),PD,tight=True,points=49)
    uptake=r['z'][-1,0,14]
    assert np.isclose(uptake,anchor,rtol=3e-5)
    check_rows.append({'Check':label+' V1 uptake recovery','Value':uptake,'Expected':anchor})
r=integrate(test_theta,tight=True,points=97)
zero=integrate(test_theta,replace(BIOLOGY,dose=0),tight=True,points=49)
assert np.max(abs(zero['kill']))<1e-12
assert np.max(abs(zero['z'][:,:,:2]-density(BIOLOGY)))<1e-9
assert r['mass']<1e-7 and np.diff(r['kill'],axis=0).min()>-1e-9
alternate=integrate(test_theta,tight=True,points=97,method='Radau')
assert np.max(abs(r['kill']-alternate['kill']))<2e-6
# Full label exchange, including populations and three-state trafficking inputs.
perm=[0,1,3,2,4]
swap=replace(BIOLOGY,target_densities=tuple(map(tuple,density(BIOLOGY)[perm,::-1])),
 ki=(BIOLOGY.ki[1],BIOLOGY.ki[0],BIOLOGY.ki[2]),k0=BIOLOGY.k0[::-1])
swapped=integrate(test_theta[[2,3,0,1]],swap,PD,tight=True,points=97)
assert np.max(abs(r['kill'][:,perm]-swapped['kill']))<2e-6
check_rows += [{'Check':'Relative antibody/payload mass error','Value':r['mass'],'Expected':0},
 {'Check':'Independent solver kill difference','Value':np.max(abs(r['kill']-alternate['kill'])),'Expected':0}]
checks=pd.DataFrame(check_rows);checks.to_csv(OUT/'implementation_checks.csv',index=False)
display(checks)
# A genuinely asymmetric relabelling check, including population-specific PD.
asym=replace(BIOLOGY,rho=(2.,.5),recycling_rates=(.2,.4,.6),
             processing_rates=(.8,.6,.3),degradation_rates=(.5,.9,.7))
ap=KillModel(c50_uM=(4.,3.,5.,6.,7.),maximum_rate_h=(.08,.05,.04,.06,.07))
th=np.log10([2e3,.3,4e4,2.])
a=integrate(th,asym,ap,tight=True,points=49)
sw=replace(asym,target_densities=tuple(map(tuple,density(asym)[perm,::-1])),
 ki=tuple(np.array(asym.ki)[[1,0,2]]),k0=asym.k0[::-1],
 recycling_rates=tuple(np.array(asym.recycling_rates)[[1,0,2]]),
 processing_rates=tuple(np.array(asym.processing_rates)[[1,0,2]]),
 degradation_rates=tuple(np.array(asym.degradation_rates)[[1,0,2]]))
sp=KillModel(c50_uM=tuple(np.array(ap.c50_uM)[perm]),maximum_rate_h=tuple(np.array(ap.maximum_rate_h)[perm]))
b=integrate(th[[2,3,0,1]],sw,sp,tight=True,points=49)
assert np.max(abs(a['kill'][:,perm]-b['kill']))<2e-6
no_release=replace(BIOLOGY,half=np.inf,processing_rates=(0.,0.,0.),degradation_rates=(0.,0.,0.))
n=integrate(th,no_release,tight=True,points=49)
assert np.max(abs(n['kill']))<1e-10 and n['mass']<1e-7
# Constant-concentration exposure has the analytic solution used by the hazard model.
from scipy.integrate import quad
h=quad(lambda t:.08*2/(4+2),0,24)[0]
assert np.isclose(-np.expm1(-h),1-np.exp(-.08*24/3))
print('Additional checks passed: asymmetric target relabelling, recycling without payload release, analytic exposure response.')


## 4. Multiobjective optimisation of tumour selectivity
For the four log binding rates $\theta$ and the listed biological inputs $\phi$, minimise jointly
$$\left(-\min_\phi K_T(\theta,\phi),\quad
\max_\phi K_H(\theta,\phi),\quad
\max_{\phi,j\in healthy}K_j(\theta,\phi)\right).$$
These are three separate objectives, without a weighted sum or a tumour/healthy ratio that can favour negligible activity. The maximum healthy-subgroup objective prevents the global average from hiding the most affected population. Extrema can occur in different input variants; all variant-specific outcomes are exported.

Retain the illustrative hard ceilings of 0.95% overall healthy kill and 10% for each healthy subgroup in every listed input variant. The subgroup ceiling is an arbitrary demonstration requirement, not from the paper or a safety standard. The synthetic input set is finite, not a confidence distribution.

Use two seeded three-objective NSGA-II searches. Refine six declared overall/subgroup budgets, with two starts per budget, by maximising tumour kill subject to each budget. This epsilon-constraint procedure samples different parts of the trade-off; it does not alter biological mechanisms. Keep intermediate candidates too.

Reintegrate every distinct screened feasible combination tightly, then remove dominated combinations. A design is dominated if another is at least as good in all three objectives and strictly better in at least one. The retained results are the **observed Pareto set**, not certified global optima. Bounds remain V1's exploratory kon 10²–10⁷ M⁻¹s⁻¹ and koff 10⁻⁵–10² s⁻¹.

In [ ]:
from joblib import Parallel, delayed, parallel_config
from scipy.stats import qmc
LOWER=np.array(BOUNDS)[:,0];UPPER=np.array(BOUNDS)[:,1]
WORKERS=4
GLOBAL_EVALUATIONS=256
LOCAL_ITERATIONS=40
# These are objective-space sampling budgets, not changes to biological mechanisms.
# Columns: overall healthy kill (%), maximum healthy subgroup kill (%).
EPSILON_BUDGETS=[(.05,1.),(.10,2.5),(.20,5.),(.40,7.5),(.95,10.),(.10,10.)]

def parallel(function,args):
    with parallel_config(backend='loky',inner_max_num_threads=1):
        return Parallel(n_jobs=WORKERS)(delayed(function)(a) for a in args)

def evaluate_design(x,tight=False):
    a=assess(x,tight=tight)
    return dict(theta=list(map(float,x)),tumour=float(a['tumour']),healthy=float(a['healthy']),
                subgroup=float(a['outcomes'][:,2:].max()),minimum_margin=float(a['margins'].min()),
                outcomes=a['outcomes'].tolist())

def objectives(rows):
    return np.array([[-r['tumour'],r['healthy'],r['subgroup']] for r in rows])

def nondominated(rows):
    if not rows:return []
    values=objectives(rows);keep=[]
    for i,v in enumerate(values):
        dominated=np.all(values<=v+1e-9,axis=1)&np.any(values<v-1e-9,axis=1)
        if not dominated.any():keep.append(rows[i])
    return keep

def global_search(seed):
    random.seed(seed);cache={};history=[]
    def ev(x):
        key=tuple(map(float,x))
        if key not in cache:
            row=evaluate_design(x);row.update(seed=seed,stage='NSGA-II');cache[key]=row;history.append(row)
        return cache[key]
    problem=Problem(4,3,1)
    problem.types[:]=[Real(*b) for b in BOUNDS];problem.constraints[:]='<=0'
    def fn(x):
        r=ev(x)
        return [-r['tumour'],r['healthy'],r['subgroup']],[-r['minimum_margin']]
    problem.function=fn
    grid=qmc.scale(qmc.LatinHypercube(4,seed=seed).random(40),LOWER,UPPER)
    grid=np.vstack([grid,[[2,1,2,1],[4,0,4,0],[2,-1,4,1],[4,1,2,-1],[2,2,2,2],[7,2,7,2],[3,0,4,1],[4,1,3,0]]])
    starts=[]
    for x in grid:
        sol=Solution(problem);sol.variables[:]=x;starts.append(sol)
    algorithm=NSGAII(problem,population_size=48,generator=InjectedPopulation(starts))
    algorithm.run(GLOBAL_EVALUATIONS)
    return {'seed':seed,'history':history,'evaluations':len(history),'algorithm_nfe':algorithm.nfe}

started=time.time()
runs=parallel(global_search,SEEDS)
(OUT/'global_searches.json').write_text(json.dumps(runs,indent=2))
history=[row for run in runs for row in run['history']]

# Epsilon-constraint refinement samples several trade-offs rather than selecting
# only the maximum-tumour endpoint. Two starts per budget, one from each seed.
def refine(task):
    overall_cap,subgroup_cap,seed,x0=task;cache={};rows=[]
    def ev(x):
        key=tuple(map(float,x))
        if key not in cache:
            r=evaluate_design(x);r.update(seed=seed,stage='epsilon refinement',budgets=[overall_cap,subgroup_cap])
            cache[key]=r;rows.append(r)
        return cache[key]
    def constraints(x):
        r=ev(x)
        return np.array([r['minimum_margin'],overall_cap-r['healthy'],subgroup_cap-r['subgroup']])-2e-4
    fit=minimize(lambda x:-ev(x)['tumour'],x0,method='SLSQP',bounds=BOUNDS,
                 constraints=[{'type':'ineq','fun':constraints}],
                 options={'maxiter':LOCAL_ITERATIONS,'ftol':1e-6,'eps':1e-3})
    ev(fit.x)
    return {'seed':seed,'overall_cap':overall_cap,'subgroup_cap':subgroup_cap,
            'success':bool(fit.success),'message':str(fit.message),'history':rows}

tasks=[]
for overall_cap,subgroup_cap in EPSILON_BUDGETS:
    for run in runs:
        candidates=[r for r in run['history'] if r['minimum_margin']>=0 and r['healthy']<=overall_cap and r['subgroup']<=subgroup_cap]
        if candidates:start=max(candidates,key=lambda r:r['tumour'])
        else:start=min(run['history'],key=lambda r:max(0,-r['minimum_margin'])+max(0,r['healthy']-overall_cap)+max(0,r['subgroup']-subgroup_cap))
        tasks.append((overall_cap,subgroup_cap,run['seed'],start['theta']))
refinements=parallel(refine,tasks)
(OUT/'refinements.json').write_text(json.dumps(refinements,indent=2))
history += [row for run in refinements for row in run['history']]
(OUT/'search_history.json').write_text(json.dumps(history))
unique={tuple(np.round(r['theta'],10)):r for r in history if r['minimum_margin']>=-2e-5}
if not unique:raise RuntimeError('No feasible design found; constraints were not relaxed.')
# Reintegrate ALL screened feasible distinct combinations, not just the apparent front.
# This prevents a screening-tolerance error from silently determining dominance.
verified=parallel(lambda row:evaluate_design(row['theta'],tight=True),list(unique.values()))
verified=[r for r in verified if r['minimum_margin']>=-1e-7]
if not verified:raise RuntimeError('No design passed tight integration; no frontier exported.')
pareto=nondominated(verified)
assert len(pareto)>0
(OUT/'verified_feasible_designs.json').write_text(json.dumps(verified,indent=2))
(OUT/'observed_pareto_designs.json').write_text(json.dumps(pareto,indent=2))
print(f'{len(history)} search evaluations; {len(verified)} tightly integrated feasible designs; {len(pareto)} observed Pareto designs; {(time.time()-started)/60:.1f} minutes.')
search_summary=pd.DataFrame([{'Overall budget (%)':r['overall_cap'],'Subgroup budget (%)':r['subgroup_cap'],
 'Seed':r['seed'],'Local termination':r['message']} for r in refinements])
display(search_summary)


## 5. Selectivity trade-offs and corresponding binding rates
Each row is one complete four-rate combination and its two derived affinities. No single point is labelled the universal best. The shortlist spans objective trade-offs; the complete observed front and per-input outcomes are exported. Do not combine marginal rate ranges independently.

The complete set includes closely spaced numerical iterates; its size is not a count of experimentally distinguishable antibodies. The six-row shortlist makes the trade-offs easier to inspect. In the affinity plot, purple indicates lower minimum tumour cell kill and yellow indicates higher minimum tumour cell kill.

In [ ]:
rate_rows=[];outcome_rows=[]
for i,r in enumerate(sorted(pareto,key=lambda r:-r['tumour'])):
    rates=10.**np.array(r['theta']);dist=np.minimum(np.array(r['theta'])-LOWER,UPPER-np.array(r['theta']))
    name=f'P{i+1:03}'
    rate_rows.append(dict(design=name,**dict(zip(RATES,rates)),KD1_uM=1e6*rates[1]/rates[0],KD2_uM=1e6*rates[3]/rates[2],
      minimum_tumour_kill_pct=r['tumour'],maximum_overall_healthy_kill_pct=r['healthy'],maximum_healthy_subgroup_kill_pct=r['subgroup'],
      minimum_constraint_margin_pp=r['minimum_margin'],near_search_boundary=bool(dist.min()<.03),
      T1_dissociation_half_s=np.log(2)/rates[1],T2_dissociation_half_s=np.log(2)/rates[3]))
    for (variant,s,p),values in zip(INPUT_SET,r['outcomes']):
        outcome_rows.append(dict(design=name,input_variant=variant,**dict(zip(
          ['tumour_kill_pct','overall_healthy_kill_pct','healthy_neither_pct','healthy_T1_pct','healthy_T2_pct','healthy_both_pct'],values))))
front=pd.DataFrame(rate_rows);outcomes=pd.DataFrame(outcome_rows)
front.to_csv(OUT/'pareto_binding_rates.csv',index=False)
outcomes.to_csv(OUT/'outcomes_by_input.csv',index=False)
# A visual shortlist spanning objective trade-offs, with duplicate points suppressed.
obj=front[['minimum_tumour_kill_pct','maximum_overall_healthy_kill_pct','maximum_healthy_subgroup_kill_pct']].values
normal=(obj-obj.min(axis=0))/np.maximum(np.ptp(obj,axis=0),1e-12)
chosen=list(dict.fromkeys([0,int(np.argmin(obj[:,1])),int(np.argmin(obj[:,2]))]))
while len(chosen)<min(6,len(front)):
    distance=np.min([np.linalg.norm(normal-normal[j],axis=1) for j in chosen],axis=0);distance[chosen]=-1
    chosen.append(int(np.argmax(distance)))
shortlist=front.iloc[chosen].sort_values('minimum_tumour_kill_pct')
shortlist.to_csv(OUT/'tradeoff_shortlist.csv',index=False)
display(shortlist[['design','minimum_tumour_kill_pct','maximum_overall_healthy_kill_pct','maximum_healthy_subgroup_kill_pct']].round(4))
display(shortlist[['design',*RATES,'KD1_uM','KD2_uM','near_search_boundary']])

# Select by explicit application requirements, not an implicit universal 'best'.
def select_from_front(minimum_tumour_pct,maximum_overall_pct,maximum_subgroup_pct):
    return front[(front.minimum_tumour_kill_pct>=minimum_tumour_pct)&
                 (front.maximum_overall_healthy_kill_pct<=maximum_overall_pct)&
                 (front.maximum_healthy_subgroup_kill_pct<=maximum_subgroup_pct)].copy()
# Demonstrate filtering without changing the optimisation objective or relaxing constraints.
example=select_from_front(20.,.2,5.)
example.to_csv(OUT/'illustrative_requirements_subset.csv',index=False)

fig,axs=plt.subplots(1,2,figsize=(11,4),layout='constrained')
all_values=objectives(verified)
axs[0].scatter(all_values[:,1],-all_values[:,0],color='grey',alpha=.12,s=10,label='Verified feasible candidates')
pts=axs[0].scatter(front.maximum_overall_healthy_kill_pct,front.minimum_tumour_kill_pct,
                  c=front.maximum_healthy_subgroup_kill_pct,cmap='viridis',s=20,label='Observed Pareto set')
axs[0].set(xlabel='Highest overall healthy cell kill (%)',ylabel='Lowest tumour cell kill (%)',title='Full-model selectivity trade-offs')
axs[0].legend(fontsize=7);fig.colorbar(pts,ax=axs[0],label='Highest healthy subgroup kill (%)')
axs[1].scatter(front.KD1_uM,front.KD2_uM,c=front.minimum_tumour_kill_pct,cmap='plasma',s=20)
for _,r in shortlist.iterrows():axs[1].annotate(r.design,(r.KD1_uM,r.KD2_uM),fontsize=7,xytext=(3,3),textcoords='offset points')
axs[1].set(xscale='log',yscale='log',xlabel='Target 1 KD (µM)',ylabel='Target 2 KD (µM)',title='Joint affinities of the Pareto designs')
fig.savefig(OUT/'selectivity_pareto.png',dpi=180);plt.show();plt.close(fig)

assert (front.minimum_constraint_margin_pp>=-1e-7).all()
assert len(nondominated(pareto))==len(pareto)
# Verify several widely separated trade-offs with a second ODE solver.
errors=[]
for index in chosen[:3]:
    theta=np.log10(front.iloc[index][RATES].to_numpy(dtype=float))
    for name,s,p in INPUT_SET:
        a=integrate(theta,s,p,tight=True,points=97)
        b=integrate(theta,s,p,tight=True,points=97,method='Radau')
        errors.append(np.max(abs(a['kill']-b['kill'])))
        assert a['mass']<1e-7 and np.diff(a['kill'],axis=0).min()>-1e-9
assert max(errors)<2e-6
summary=dict(pareto_designs=len(front),verified_feasible_designs=len(verified),
 maximum_minimum_tumour_kill_pct=float(front.minimum_tumour_kill_pct.max()),
 lowest_overall_healthy_kill_pct=float(front.maximum_overall_healthy_kill_pct.min()),
 highest_tumour_design_overall_healthy_pct=float(front.iloc[0].maximum_overall_healthy_kill_pct),
 highest_tumour_design_subgroup_pct=float(front.iloc[0].maximum_healthy_subgroup_kill_pct),
 boundary_designs=int(front.near_search_boundary.sum()),cross_solver_difference=float(max(errors)),
 example_requirement_subset_count=len(example),
 interpretation='Observed three-objective Pareto set under synthetic full-model inputs; not a proven global frontier or actual affinity recommendation.')
(OUT/'summary.json').write_text(json.dumps(summary,indent=2))
display(Markdown(f'''**Result:** {len(front)} binding-rate combinations are nondominated among {len(verified)} tightly integrated feasible candidates. No evaluated feasible candidate improves all three objectives over one of these without a trade-off. This is an **observed Pareto set**, not proof of the global frontier.

The highest minimum tumour cell kill found is **{front.minimum_tumour_kill_pct.max():.2f}%**. Other designs trade tumour activity for lower overall or subgroup healthy-cell kill. All biological inputs are synthetic or inherited reconstruction assumptions; these are method outputs, not recommended affinities.

**Selection:** apply intended requirements to the Pareto table. The illustrative requirement of at least 20% tumour kill, at most 0.2% overall healthy kill, and at most 5% subgroup kill retains **{len(example)}** sampled designs. These cutoffs only demonstrate the interface; they are not biological recommendations. If a filter returns no rows, report that no qualifying design was found in the sampled set.

**Limits:** {int(front.near_search_boundary.sum())} Pareto designs approach the inherited search bounds. The displayed scatter is a projection of a three-objective set, so some points can overlap or appear dominated in two dimensions. Low-toxicity designs may also have insufficient tumour activity; Pareto membership alone does not make a design useful.'''))


## 6. Use the method with real inputs
Replace the synthetic target/trafficking/payload inputs and calibrate the exposure-to-cell-kill relationship from suitable time-course data. Set biological uncertainty, healthy-cell limits, treatment schedule, and practical binding-rate bounds explicitly. Run the same multiobjective search, then filter the resulting joint designs using the required tumour activity and healthy-cell limits.

Low activity can be Pareto-efficient because it has low toxicity; membership alone is not a useful-molecule criterion. An empty filtered set means no qualifying design was found, not proof of impossibility. Multiple near-equivalent rates and search-boundary solutions remain possible. The method does not predict sequence-level engineering feasibility or clinical safety.

The full model retains finite-horizon exposure scoring without cell-depletion feedback to PK, one pooled extracellular payload compartment, and V1's surface-binding closure. These assumptions require assessment for the intended experimental system.

Source: Stoddard et al. (2026), doi:10.20935/AcadDrug8407; V1 reconstruction and V4 combined ODE implementation. Missing inputs are LLM-assigned synthetic assumptions, not experimental data. This notebook supersedes the reduced analytical objective for the purpose of tumour-selective binding optimisation.

**Run-specific search limitation:** the lowest overall healthy-cell kill among the evaluated feasible designs was 0.1170%. Consequently, the tighter 0.05% and 0.10% overall budgets had no qualifying design in this evaluated set. Failed or iteration-limited refinements are retained in the search log. This does not prove those budgets are impossible.